In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import product

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    balanced_accuracy_score
)

In [ ]:
#rutas
TRAIN_NPZ_PATH = "/content/drive/MyDrive/tesis/train_3500_40_1.npz"
TEST_NPZ_PATH  = "/content/drive/MyDrive/tesis/test_3500_40_1.npz"

SPLIT_SAVE_PATH = "/content/drive/MyDrive/tesis/split_1.npz"

LABEL_ORDER = [1, 2, 3, 4, 5]
CLASS_NAMES = ["Hepatocito", "Estrellada", "Kupffer", "Endotelial", "Otras"]

RANDOM_STATE = 42
VAL_SIZE = 0.10

OUTPUT_DIR = "/content/drive/MyDrive/tesis/randomforest_pipeline_pca_macro_1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULTS_CSV_PATH     = os.path.join(OUTPUT_DIR, "rf_hyperparameter_search_results.csv")
BEST_PARAMS_TXT_PATH = os.path.join(OUTPUT_DIR, "best_params_randomforest.txt")
MODEL_PATH           = os.path.join(OUTPUT_DIR, "randomforest_pipeline.pkl")

METRICS_CSV_PATH     = os.path.join(OUTPUT_DIR, "metrics_summary_randomforest.csv")
METRICS_TXT_PATH     = os.path.join(OUTPUT_DIR, "metrics_report_randomforest.txt")
METRICS_PNG_PATH     = os.path.join(OUTPUT_DIR, "metrics_summary_randomforest.png")

CM_VAL_PATH          = os.path.join(OUTPUT_DIR, "confusion_matrix_validacion_randomforest.png")
CM_TEST_PATH         = os.path.join(OUTPUT_DIR, "confusion_matrix_test_randomforest.png")

IMPORTANCE_CSV_PATH  = os.path.join(OUTPUT_DIR, "top20_component_importance_randomforest.csv")
IMPORTANCE_PNG_PATH  = os.path.join(OUTPUT_DIR, "top20_component_importance_randomforest.png")

print("Carpeta de salida:", OUTPUT_DIR)

In [ ]:
from sklearn.model_selection import train_test_split

def load_npz_data(npz_path):
    data = np.load(npz_path, allow_pickle=True)

    print(f"\nContenido de {npz_path}:")
    print(data.files)

    X = data["embeddings"]
    y = data["labels"]
    paths = data["paths"]

    print(f"embeddings shape: {X.shape}")
    print(f"labels shape: {y.shape}")
    print(f"paths shape: {paths.shape}")
    print(f"clases únicas: {np.unique(y)}")

    return X, y, paths


X_full, y_full, paths_full = load_npz_data(TRAIN_NPZ_PATH)
X_test, y_test, paths_test = load_npz_data(TEST_NPZ_PATH)

X_train, X_val, y_train, y_val, paths_train, paths_val = train_test_split(
    X_full,
    y_full,
    paths_full,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_full
)

print("\n==============================")
print("Shapes después del split")
print("==============================")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val  :", X_val.shape)
print("y_val  :", y_val.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

np.savez(
    SPLIT_SAVE_PATH,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test
)

print("Split guardado correctamente en:")
print(SPLIT_SAVE_PATH)

In [ ]:
def print_class_counts(y, split_name):
    unique, counts = np.unique(y, return_counts=True)
    print(f"\nConteo por clase - {split_name}")
    for u, c in zip(unique, counts):
        print(f"Clase {u}: {c}")

print_class_counts(y_train, "Train")
print_class_counts(y_val, "Validación")
print_class_counts(y_test, "Test")

In [ ]:
def evaluate_model(y_true, y_pred, split_name):
    acc = accuracy_score(y_true, y_pred)

    precision_w = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall_w    = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1_w        = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    precision_m = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall_m    = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_m        = f1_score(y_true, y_pred, average="macro", zero_division=0)

    bal_acc = balanced_accuracy_score(y_true, y_pred)

    report_txt = classification_report(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )

    print("\n" + "=" * 70)
    print(f"RESULTADOS - {split_name.upper()}")
    print("=" * 70)
    print(f"Accuracy           : {acc:.4f}")
    print(f"Precision weighted : {precision_w:.4f}")
    print(f"Recall weighted    : {recall_w:.4f}")
    print(f"F1-score weighted  : {f1_w:.4f}")
    print(f"Precision macro    : {precision_m:.4f}")
    print(f"Recall macro       : {recall_m:.4f}")
    print(f"F1-score macro     : {f1_m:.4f}")
    print(f"Balanced Accuracy  : {bal_acc:.4f}")

    print(f"\nReporte por clase - {split_name}")
    print(report_txt)

    metrics_dict = {
        "split": split_name,
        "accuracy": acc,
        "precision_weighted": precision_w,
        "recall_weighted": recall_w,
        "f1_weighted": f1_w,
        "precision_macro": precision_m,
        "recall_macro": recall_m,
        "f1_macro": f1_m,
        "balanced_accuracy": bal_acc
    }

    return metrics_dict, report_txt


def save_confusion_matrix(y_true, y_pred, split_name, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)

    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, cmap="Blues", values_format="d", xticks_rotation=45, colorbar=False)
    plt.title(f"Matriz de confusión - {split_name}")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print("Guardado:", save_path)


def save_metrics_table_png(metrics_df, save_path):
    df_show = metrics_df.copy().round(4)

    fig, ax = plt.subplots(figsize=(12, 2 + 0.5 * len(df_show)))
    ax.axis("off")

    table = ax.table(
        cellText=df_show.values,
        colLabels=df_show.columns,
        loc="center",
        cellLoc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)

    plt.title("Resumen de métricas - Random Forest", pad=20)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print("Guardado:", save_path)

In [ ]:
param_grid = {
    "pca__n_components": [0.95, 256],
    "pca__whiten": [False],
    "rf__n_estimators": [200, 300],
    "rf__max_depth": [10, 15, 20],
    "rf__min_samples_split": [5, 10],
    "rf__min_samples_leaf": [2, 4, 6],
    "rf__max_features": ["sqrt", "log2"]
}

all_combinations = list(product(
    param_grid["pca__n_components"],
    param_grid["pca__whiten"],
    param_grid["rf__n_estimators"],
    param_grid["rf__max_depth"],
    param_grid["rf__min_samples_split"],
    param_grid["rf__min_samples_leaf"],
    param_grid["rf__max_features"]
))

print(f"Total de combinaciones a evaluar: {len(all_combinations)}")

results = []
best_score = -1
best_params = None
best_model = None
best_loss = 1e30

for i, (
    n_components,
    whiten,
    n_estimators,
    max_depth,
    min_samples_split,
    min_samples_leaf,
    max_features
) in enumerate(all_combinations, start=1):

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(
            n_components=n_components,
            whiten=whiten,
            random_state=RANDOM_STATE
        )),
        ("rf", RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            bootstrap=False,
            n_jobs=-1
        ))
    ])

    model.fit(X_train, y_train)

    y_val_pred = model.predict(X_val)
    y_val_proba = model.predict_proba(X_val)

    acc_val = accuracy_score(y_val, y_val_pred)
    f1_w = f1_score(y_val, y_val_pred, average="weighted", zero_division=0)
    f1_m = f1_score(y_val, y_val_pred, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(y_val, y_val_pred)
    loss1 = log_loss(y_val, y_val_proba)

    results.append({
        "pca__n_components": n_components,
        "pca__whiten": whiten,
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "min_samples_split": min_samples_split,
        "min_samples_leaf": min_samples_leaf,
        "max_features": max_features,
        "accuracy_val": acc_val,
        "f1_weighted_val": f1_w,
        "f1_macro_val": f1_m,
        "balanced_acc_val": bal_acc,
        "log_loss_val": loss1
    })

    print(
        f"[{i}/{len(all_combinations)}] "
        f"pca__n_components={n_components}, n_estimators={n_estimators}, "
        f"max_depth={max_depth}, min_samples_split={min_samples_split}, "
        f"min_samples_leaf={min_samples_leaf}, max_features={max_features} --> "
        f"F1 macro val = {f1_m:.4f}, log loss val = {loss1:.4f}"
    )

    # Selección principal por F1 macro
    # Desempate secundario por log_loss más bajo
    if (f1_m > best_score) or (f1_m == best_score and loss1 < best_loss):
        best_score = f1_m
        best_loss = loss1
        best_params = {
            "pca__n_components": n_components,
            "pca__whiten": whiten,
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "min_samples_split": min_samples_split,
            "min_samples_leaf": min_samples_leaf,
            "max_features": max_features
        }
        best_model = model

print("\nMejores hiperparámetros encontrados:")
print(best_params)
print(f"Mejor F1 macro en validación: {best_score:.4f}")
print(f"Log loss asociado: {best_loss:.4f}")

results_df = pd.DataFrame(results).sort_values(by="f1_macro_val", ascending=False)
results_df = results_df.reset_index(drop=True)

print("\nTop 10 combinaciones:")
print(results_df.head(10))

results_df.to_csv(RESULTS_CSV_PATH, index=False)
print("Archivo guardado:", RESULTS_CSV_PATH)

In [ ]:
rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(
        n_components=best_params["pca__n_components"],
        whiten=best_params["pca__whiten"],
        random_state=RANDOM_STATE
    )),
    ("rf", RandomForestClassifier(
        n_estimators=best_params["n_estimators"],
        max_depth=best_params["max_depth"],
        min_samples_split=best_params["min_samples_split"],
        min_samples_leaf=best_params["min_samples_leaf"],
        max_features=best_params["max_features"],
        class_weight="balanced",
        random_state=RANDOM_STATE,
        bootstrap=False,
        n_jobs=-1
    ))
])

print("\nEntrenando Random Forest final...")
rf_pipeline.fit(X_train, y_train)

y_train_pred = rf_pipeline.predict(X_train)
y_val_pred   = rf_pipeline.predict(X_val)
y_test_pred  = rf_pipeline.predict(X_test)

train_metrics, train_report = evaluate_model(y_train, y_train_pred, "Train")
val_metrics, val_report     = evaluate_model(y_val, y_val_pred, "Validación")
test_metrics, test_report   = evaluate_model(y_test, y_test_pred, "Test")

metrics_df = pd.DataFrame([train_metrics, val_metrics, test_metrics])

print("\nResumen de métricas:")
print(metrics_df.round(4))

In [ ]:
save_confusion_matrix(y_val, y_val_pred, "Validación", CM_VAL_PATH)
save_confusion_matrix(y_test, y_test_pred, "Test", CM_TEST_PATH)

In [ ]:
rf_model = rf_pipeline.named_steps["rf"]
pca_model = rf_pipeline.named_steps["pca"]

importances = rf_model.feature_importances_

n_components_used = len(importances)
component_names = [f"PC_{i+1:03d}" for i in range(n_components_used)]

top_k = min(20, n_components_used)
top_idx = np.argsort(importances)[::-1][:top_k]

print("\nTop 20 componentes principales más importantes:")
for rank, idx in enumerate(top_idx, start=1):
    print(f"{rank:02d}. {component_names[idx]} -> {importances[idx]:.6f}")

importance_df = pd.DataFrame({
    "component": [component_names[idx] for idx in top_idx],
    "importance": importances[top_idx]
})

importance_df.to_csv(IMPORTANCE_CSV_PATH, index=False)
print("\nArchivo guardado:", IMPORTANCE_CSV_PATH)

plt.figure(figsize=(10, 6))
plt.barh(importance_df["component"][::-1], importance_df["importance"][::-1])
plt.title("Top 20 componentes principales más importantes - Random Forest")
plt.xlabel("Importancia")
plt.ylabel("Componente principal")
plt.tight_layout()
plt.savefig(IMPORTANCE_PNG_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Gráfico guardado:", IMPORTANCE_PNG_PATH)

In [ ]:
metrics_df.to_csv(METRICS_CSV_PATH, index=False)
save_metrics_table_png(metrics_df, METRICS_PNG_PATH)

with open(BEST_PARAMS_TXT_PATH, "w", encoding="utf-8") as f:
    f.write("MEJORES HIPERPARÁMETROS - Random Forest\n")
    f.write("=" * 80 + "\n")
    for k, v in best_params.items():
        f.write(f"{k}: {v}\n")
    f.write(f"\nMejor F1 macro en validación: {best_score:.6f}\n")
    f.write(f"Log loss asociado: {best_loss:.6f}\n")

with open(METRICS_TXT_PATH, "w", encoding="utf-8") as f:
    f.write("REPORTE COMPLETO - Random Forest + Scaler + PCA\n")
    f.write("=" * 80 + "\n\n")

    f.write("Shapes\n")
    f.write("-" * 80 + "\n")
    f.write(f"X_train: {X_train.shape} | y_train: {y_train.shape}\n")
    f.write(f"X_val:   {X_val.shape} | y_val:   {y_val.shape}\n")
    f.write(f"X_test:  {X_test.shape} | y_test:  {y_test.shape}\n\n")

    f.write("Mejores hiperparámetros\n")
    f.write("-" * 80 + "\n")
    for k, v in best_params.items():
        f.write(f"{k}: {v}\n")
    f.write(f"\nMejor F1 macro en validación: {best_score:.6f}\n")
    f.write(f"Log loss asociado: {best_loss:.6f}\n\n")

    f.write("Resumen numérico de métricas\n")
    f.write("-" * 80 + "\n")
    f.write(metrics_df.to_string(index=False))
    f.write("\n\n")

    f.write("Classification Report - Train\n")
    f.write("-" * 80 + "\n")
    f.write(train_report)
    f.write("\n\n")

    f.write("Classification Report - Validación\n")
    f.write("-" * 80 + "\n")
    f.write(val_report)
    f.write("\n\n")

    f.write("Classification Report - Test\n")
    f.write("-" * 80 + "\n")
    f.write(test_report)
    f.write("\n\n")

joblib.dump(rf_pipeline, MODEL_PATH)

print("\nArchivos guardados correctamente:")
print(SPLIT_SAVE_PATH)
print(RESULTS_CSV_PATH)
print(BEST_PARAMS_TXT_PATH)
print(MODEL_PATH)
print(METRICS_CSV_PATH)
print(METRICS_TXT_PATH)
print(METRICS_PNG_PATH)
print(CM_VAL_PATH)
print(CM_TEST_PATH)
print(IMPORTANCE_CSV_PATH)
print(IMPORTANCE_PNG_PATH)

In [ ]:
print("Mejores hiperparámetros finales:")
print(best_params)

print("\nTop 10 combinaciones por F1 macro:")
print(results_df.head(10))